# Delete Exposures and Nights

Interactively delete exposures and nights of observations after inspect_nights.

Author: Qiushi Chris Tian

Created: 2026-09-28

<!-- Updated: -->

## Preamble

In [ ]:
# %matplotlib ipympl

In [ ]:
import statistics
from dep import plt, va
import numpy as np
from pathlib import Path
from astropy.table import Table
from newport import get_input_path, TARGET_GAIA_DR3, BAND_ORDER_INDEX

In [ ]:
input_path = get_input_path(TARGET)
print(f'Reading original table from: {input_path}\n')

t = Table.read(input_path)
print(f'Original length: {len(t)}\n')

## Setting/Config

In [ ]:
TARGET = 'TOI-1410'

WHOLE_NIGHTS = []
BAND_NIGHTS = {}
JDS = []
DATETIMES = ['2023-09-07T04:10:29.989', '2023-09-07T04:10:57.460', '2023-09-07T04:11:24.750']

print(
    f'{len(WHOLE_NIGHTS)} whole night(s), '
    f'{len(BAND_NIGHTS)} band night(s), '
    f'{len(JDS)} individual exposure(s) from JD, and '
    f'{len(DATETIMES)} individual exposure(s) from datetime'
)

PREVIEW_COLS = ['band', 'night', 'time', 'jd', 'airmass', 'exptime', TARGET_GAIA_DR3[TARGET]]#, 'skytemp']

## Inspect Rows in Table

In [ ]:
t[t['night'] > 20240000]['airmass'].min()

In [ ]:
_, ax = plt.subplots()
va.hist(
    t[t['night'] > 20240000]['airmass'],
    xlog=False, ylog=False, ax=ax, colors='red'
)
va.hist(
    t[t['night'] < 20240000]['airmass'],
    xlog=False, ylog=False, ax=ax, colors='green'
)

In [ ]:
pt = t[PREVIEW_COLS][(t['night'] == 20230906) & (t['band'] == 'R')]
pt.sort('jd')
pt.pprint_all()

In [ ]:
bt = Table.read('data/tables/opt_comp_stars/TOI-1410/20260930T21_man_del_c8/unbin_results_I.fits')

In [ ]:
exptimes = np.unique(bt['exptime'])

for et in exptimes:
    _t = bt[bt['exptime'] == et]
    nights = np.unique(_t['night'])
    nights.sort()
    print(et, nights)

#   10 11 12 17 18 19
# b m  ml l  h  l  l
# v 3 2.5 1  4  1  1
# r h  lh l  m  -  l
# i h  lh l  m  -  l

In [ ]:
ft = Table.read('data/tables/opt_comp_stars/TOI-1410/20260930T21_man_del_c8/unbin_results_all.fits')

In [ ]:
for b in np.unique(ft['band']):
    print(b)
    _t = ft[ft['band'] == b]
    m = statistics.mode(_t['exptime'])
    st = _t[_t['exptime'] != m]
    st.sort('jd')
    print(np.unique(st['night']))
    print()

In [ ]:
_ = bt[bt['night'] == 20230711]
_.sort('jd')
_

In [ ]:
pvc = PREVIEW_COLS[1:]

bts = bt[bt['night'] == 20230906]
bts.sort('jd')
bts.pprint_all()

## Building Mask

In [ ]:
del_mask = np.zeros(len(t), dtype=bool)

for wn in WHOLE_NIGHTS:
    selected = t['night'] == wn
    print(f'{np.sum(selected)} exposure(s) on {wn} ...')
    del_mask |= selected

for night, bands in BAND_NIGHTS.items():
    for band in bands:
        selected = (t['band'] == band) & (t['night'] == night)
        print(f'{np.sum(selected)} exposure(s) in {band} band on {night} ...')
        del_mask |= selected

for jd in JDS:
    selected = t['jd'] == jd
    print(f'1 exposure on {t[selected]["night"][0]} ...')
    del_mask |= selected

for dt in DATETIMES:
    selected = t['time'] == dt
    print(f'1 exposure on {t[selected]["night"][0]} ...')
    del_mask |= selected

print('Done!')

## Preview Deletion

In [ ]:
dt = t[del_mask]

print(f'Number of exposure(s): {len(dt)}')
print(f'Unique night(s): {len(np.unique(dt['night']))}\n')

dt[PREVIEW_COLS].pprint_all()

print(f'\nAfter deletion, length will become {len(t) - len(dt)}')

## Executing Deletion

In [ ]:
new_t = t[~del_mask]

print(f'New length: {len(new_t)}')

## Save Table

In [ ]:
save_path = f'data/tables/opt_comp_stars/{TARGET}/{input_path.stem}_manual_del{input_path.suffix}'
print(f'Modified table will be saved to: {save_path}\n')

In [ ]:
if not Path(save_path).exists() or input(f'{save_path} exists. Overwrite? (y/*)').casefold().startswith('y'):
    new_t.write(save_path, overwrite=True)
    print(f'Saved to {save_path}')
else:
    print(f'Saving aborted')